In [1]:
import os
import sys
import math

import random
import datetime as dt
import numpy as np
import pandas as pd
import xarray as xr
# Standard Python + data analysis packages



import pydeltasnow as pyds

import matplotlib.pyplot as plt

from scipy.optimize import differential_evolution

# 1. Load datasets

In [2]:
train_data = '/Users/jakobwerkgarner/code/mt_dsnow/calibration/calibration_py/DE_rsme_comb/splited_data/Win21_all_train.nc'
val_data = '/Users/jakobwerkgarner/code/mt_dsnow/calibration/calibration_py/DE_rsme_comb/splited_data/Win21_all_validation.nc'

train_data = xr.open_dataset(train_data).drop_vars('season')
val_data = xr.open_dataset(val_data).drop_vars('season')




In [3]:
stations = train_data.station.values
every_second_station = stations[::2]

train_data = train_data.sel(station=every_second_station)

print(f"Selected {train_data.sizes['station']} stations out of {len(stations)} total.")

Selected 8 stations out of 15 total.


# 2. Prepare datasets

In [4]:
# train_data = (
#     train_data
#     .drop_vars(["HS_meas", "HSMeas"], errors="ignore")
#     .rename({"HS_mod": "HS"})
# )

# val_data = (
#     val_data
#     .drop_vars(["HS_meas", "HSMeas"], errors="ignore")
#     .rename({"HS_mod": "HS"})
# )


# Set first tiime step to 0 # Set first timestep to 0 for all stations
train_data["HS"][{"time": 0}] = 0.0

# Set first tiime step to 0 # Set first timestep to 0 for all stations
val_data["HS"][{"time": 0}] = 0.0

In [5]:
first_station = train_data.station.values[0]
hs_series = train_data["HS"].sel(station=first_station).to_pandas()
swe_series = train_data["SWE"].sel(station=first_station).to_pandas()

print("Station:", first_station)
print("HS len:", len(hs_series), "SWE len:", len(swe_series))
print("HS finite:", np.isfinite(hs_series.values).sum(), "of", len(hs_series))
print("SWE finite:", np.isfinite(swe_series.values).sum(), "of", len(swe_series))
print("HS negatives:", np.sum(hs_series.values < 0))
print("SWE negatives:", np.sum(swe_series.values < 0))
print("HS duplicate timestamps:", hs_series.index.duplicated().sum())
print("SWE duplicate timestamps:", swe_series.index.duplicated().sum())
print("HS max:", np.nanmax(hs_series.values), "SWE max:", np.nanmax(swe_series.values))

Station: Davos_Flueelastr
HS len: 6880 SWE len: 6880
HS finite: 1097 of 6880
SWE finite: 25 of 6880
HS negatives: 0
SWE negatives: 0
HS duplicate timestamps: 0
SWE duplicate timestamps: 0
HS max: 1.75 SWE max: 327.0



# 3. Define optimization problem


In [6]:
param_names = ["rho_max", "rho_null", "c_ov", "k_ov", "k", "tau", "eta_null"]

bounds = [
    (300.0, 600.0),        # rho_max
    (60.0, 200.0),         # rho_null
    (1e-5, 5e-3),          # c_ov
    (0.05, 1.0),           # k_ov
    (1e-3, 0.2),           # k
    (1e-4, 0.2),           # tau
    (1e5, 2e7),            # eta_null
]


# 4. Error metric


In [7]:

def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

In [8]:
# Pre-compute mean observed SWE and rho_bulk for normalizing the combined score
# (same approach as the R calibration script)
all_swe = []
all_rho = []
for st in train_data.station.values:
    swe = train_data["SWE"].sel(station=st).values
    hs = train_data["HS"].sel(station=st).values
    mask_swe = np.isfinite(swe)
    all_swe.extend(swe[mask_swe])
    mask_rho = mask_swe & (hs > 0)
    all_rho.extend(swe[mask_rho] / hs[mask_rho])

mean_swe_obs = np.mean(all_swe)
mean_rho_obs = np.mean(all_rho)
print(f"Normalization: mean_swe_obs={mean_swe_obs:.4f}, mean_rho_obs={mean_rho_obs:.4f}")

Normalization: mean_swe_obs=165.4310, mean_rho_obs=269.9841


In [9]:

def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))


def split_into_seasons(df):
    """
    Split a DataFrame with a DatetimeIndex into seasons running
    from August 1 to July 31.
    """
    season_year = np.where(df.index.month >= 8, df.index.year, df.index.year - 1)
    return [g for _, g in df.groupby(season_year)]


def _evaluate_season(season_df, params):
    """
    Run DeltaSNOW on one single season and return (e_swe, e_rho).
    """
    hs = season_df["hs"].copy()
    swe_obs = season_df["obs"].copy()

    hs = hs[hs.notna()]
    if hs.empty or len(hs) < 10:
        return np.inf, np.inf

    # Remove duplicate timestamps
    hs = hs[~hs.index.duplicated(keep="first")]
    swe_obs = swe_obs[~swe_obs.index.duplicated(keep="first")]

    # Ensure series starts with 0 (required by pydeltasnow)
    if hs.iloc[0] != 0.0:
        hs.iloc[0] = 0.0

    try:
        swe_pred = pyds.swe_deltasnow(
            hs,
            rho_max=params["rho_max"],
            rho_null=params["rho_null"],
            c_ov=params["c_ov"],
            k_ov=params["k_ov"],
            k=params["k"],
            tau=params["tau"],
            eta_null=params["eta_null"],
            hs_input_unit="m",
            swe_output_unit="mm",
        )
    except Exception:
        return np.inf, np.inf

    aligned = pd.concat(
        [swe_obs.rename("obs"), swe_pred.rename("pred"), hs.rename("hs")],
        axis=1,
    )

    aligned = aligned.loc[aligned["obs"].notna() & (aligned["obs"] != 0)]

    if aligned.empty:
        return np.inf, np.inf

    e_swe = rmse(aligned["obs"].values, aligned["pred"].values)

    mask = aligned["hs"].values > 0
    if mask.any():
        rho_obs = aligned["obs"].values[mask] / aligned["hs"].values[mask]
        rho_mod = aligned["pred"].values[mask] / aligned["hs"].values[mask]
        e_rho = rmse(rho_obs, rho_mod)
    else:
        e_rho = np.inf

    return e_swe, e_rho


def build_station_season_cache(ds):
    """Precompute station season data once to avoid repeated conversion in objective."""
    cache = {}
    for st in ds.station.values:
        hs = ds["HS"].sel(station=st).to_pandas()
        swe_obs = ds["SWE"].sel(station=st).to_pandas()

        station_df = pd.concat(
            [swe_obs.rename("obs"), hs.rename("hs")],
            axis=1,
        ).sort_index()

        cache[st] = split_into_seasons(station_df)
    return cache


def _eval_station_seasons(season_dfs, params):
    """
    Evaluate one station from precomputed season splits and average errors.
    """
    swe_errors = []
    rho_errors = []

    for season_df in season_dfs:
        e_swe, e_rho = _evaluate_season(season_df, params)

        if np.isfinite(e_swe):
            swe_errors.append(e_swe)
        if np.isfinite(e_rho):
            rho_errors.append(e_rho)

    if not swe_errors:
        return np.inf, np.inf

    return float(np.mean(swe_errors)), float(np.mean(rho_errors))


def evaluate_dataset(ds, params):
    results = []
    for st in ds.station.values:
        hs = ds["HS"].sel(station=st).to_pandas()
        swe_obs = ds["SWE"].sel(station=st).to_pandas()
        station_df = pd.concat(
            [swe_obs.rename("obs"), hs.rename("hs")],
            axis=1,
        ).sort_index()
        season_dfs = split_into_seasons(station_df)
        results.append(_eval_station_seasons(season_dfs, params))

    swe_errors = [e_swe for e_swe, _ in results if np.isfinite(e_swe)]
    rho_errors = [e_rho for _, e_rho in results if np.isfinite(e_rho)]

    return {
        "swe_rmse": float(np.mean(swe_errors)) if swe_errors else np.inf,
        "rho_rmse": float(np.mean(rho_errors)) if rho_errors else np.inf,
    }


# 6. Objective function

In [ ]:
lower = np.array([b[0] for b in bounds])
upper = np.array([b[1] for b in bounds])

# Build once so objective does not keep converting xarray -> pandas every call
train_station_cache = build_station_season_cache(train_data)

# Counter for tracking objective evaluations
objective_call_count = {}
objective_crash_count = {}

def denormalize(x_norm):
    """Map x from [0, 1] back to physical parameter space."""
    return lower + x_norm * (upper - lower)

def objective(x_norm, station):
    """Objective for a single station: 50/50 normalized RMSE (like R script)."""
    try:
        x_phys = denormalize(x_norm)
        params = dict(zip(param_names, x_phys))
        swe_rmse, rho_rmse = _eval_station_seasons(train_station_cache[station], params)
        
        # Validate outputs
        if not np.isfinite(swe_rmse) or not np.isfinite(rho_rmse):
            return 1e6  # Large penalty for invalid results
        
        score = 0.5 * (swe_rmse / mean_swe_obs) + 0.5 * (rho_rmse / mean_rho_obs)
        
        # Clamp score to prevent overflow
        score = min(score, 1e6)
        
        # Track calls
        if station not in objective_call_count:
            objective_call_count[station] = 0
        objective_call_count[station] += 1
        
        if objective_call_count[station] % 10 == 0 or objective_call_count[station] == 1:
            print(f"  [Call {objective_call_count[station]:3d}] SWE_RMSE={swe_rmse:8.2f} mm, rho_RMSE={rho_rmse:8.2f} kg/m\u00b3, Score={score:.6f}")
        
        return score
    except Exception as e:
        if station not in objective_crash_count:
            objective_crash_count[station] = 0
        objective_crash_count[station] += 1
        print(f"  [CRASH {objective_crash_count[station]}] Exception in objective: {type(e).__name__}: {str(e)[:100]}")
        return 1e6  # Return penalty for crashes

: 

# 7. Run Differential Evolution

In [ ]:
# DIAGNOSTIC: Simple grid search on first station (avoids differential_evolution crash)
print("Testing first station with simple grid search...\n")

test_station = all_stations[0]
print(f"Test station: {test_station}\n")

# Create a coarse grid of parameter combinations
n_samples = 4
test_params_list = []
for i in range(n_samples):
    x_rand = np.random.uniform(0, 1, len(bounds))
    x_phys = denormalize(x_rand)
    test_params_list.append(dict(zip(param_names, x_phys)))

try:
    print(f"Evaluating {n_samples} random parameter sets...")
    objective_call_count[test_station] = 0
    objective_crash_count[test_station] = 0
    
    results = []
    for j, params in enumerate(test_params_list):
        score = objective(np.random.uniform(0, 1, len(bounds)), test_station)
        results.append(score)
        print(f"  Sample {j+1}/{n_samples}: Score={score:.6f}")
    
    min_score = min(results)
    print(f"\n✓ Grid search passed!")
    print(f"  Best score: {min_score:.6f}")
    print(f"  Total objective calls: {objective_call_count[test_station]}")
    print(f"  Crashes: {objective_crash_count[test_station]}")
    print(f"\n  If crashes > 0, pydeltasnow may crash on edge cases.")
except Exception as e:
    print(f"\n✗ Grid search FAILED: {type(e).__name__}: {str(e)[:200]}")
    print("Kernel likely to crash on optimization. Check pydeltasnow stability.")

In [ ]:
import gc
from scipy.optimize import minimize

norm_bounds = [(0.0, 1.0)] * len(bounds)
all_stations = train_data.station.values
all_results = {}

print(f"Starting optimization for {len(all_stations)} stations (using minimize)...\n")

for i, station in enumerate(all_stations):
    print(f"\n{'='*70}")
    print(f"Progress: Station {i+1}/{len(all_stations)} - {station}")
    print(f"{'='*70}")
    
    objective_call_count[station] = 0
    objective_crash_count[station] = 0
    
    try:
        # Start from midpoint of parameter space
        x0 = np.full(len(bounds), 0.5)
        
        # Use L-BFGS-B (gradient-free, box-constrained)
        result = minimize(
            objective,
            x0,
            args=(station,),
            method='L-BFGS-B',
            bounds=norm_bounds,
            options={'maxiter': 50, 'ftol': 1e-4, 'disp': False},
        )

        best_params = dict(zip(param_names, denormalize(result.x)))
        all_results[station] = {
            "best_params": best_params,
            "fun": result.fun,
            "success": result.success,
            "nit": result.nit,
        }
        print(f"\nStation {station} COMPLETE")
        print(f"Total objective calls: {objective_call_count[station]}")
        print(f"Crashes encountered: {objective_crash_count[station]}")
        print(f"Best combined RMSE: {result.fun:.6f}")
        print(f"Success: {result.success}, Iterations: {result.nit}")
        print("Best parameters:")
        for name, value in best_params.items():
            print(f"  {name:12s} = {value:12.6g}")
    except Exception as e:
        print(f"\n*** STATION OPTIMIZATION FAILED ***")
        print(f"Error: {type(e).__name__}: {str(e)[:200]}")
        print(f"Skipping station {station}")
    
    gc.collect()

Starting optimization for 8 stations...


Progress: Station 1/8 - Davos_Flueelastr


# 8. Best parameters


In [ ]:
# Summary table of per-station results
rows = []
for station, res in all_results.items():
    row = {"station": station, "combined_rmse": res["fun"], "nit": res["nit"]}
    row.update(res["best_params"])
    rows.append(row)

results_df = pd.DataFrame(rows).set_index("station")
results_df

KeyError: "None of ['station'] are in the columns"

# 9. Final evaluation

In [ ]:
# Evaluate each station's best params on train and validation sets
eval_rows = []
for station, res in all_results.items():
    params = res["best_params"]
    ds_train_st = train_data.sel(station=[station])
    train_res = evaluate_dataset(ds_train_st, params)

    # Only evaluate on val if station exists there
    if station in val_data.station.values:
        ds_val_st = val_data.sel(station=[station])
        val_res = evaluate_dataset(ds_val_st, params)
    else:
        val_res = {"swe_rmse": np.nan, "rho_rmse": np.nan}

    # Combined normalized RMSE
    train_comb = 0.5 * (train_res["swe_rmse"] / mean_swe_obs) + 0.5 * (train_res["rho_rmse"] / mean_rho_obs)
    val_comb = 0.5 * (val_res["swe_rmse"] / mean_swe_obs) + 0.5 * (val_res["rho_rmse"] / mean_rho_obs)

    eval_rows.append({
        "station": station,
        "train_swe_rmse": train_res["swe_rmse"],
        "train_rho_rmse": train_res["rho_rmse"],
        "train_combined": train_comb,
        "val_swe_rmse": val_res["swe_rmse"],
        "val_rho_rmse": val_res["rho_rmse"],
        "val_combined": val_comb,
    })

eval_df = pd.DataFrame(eval_rows).set_index("station")
print("Per-station evaluation:")
print(eval_df.to_string())
print(f"\nMean train SWE RMSE:      {eval_df['train_swe_rmse'].mean():.4f} mm")
print(f"Mean train rho RMSE:      {eval_df['train_rho_rmse'].mean():.4f} kg/m³")
print(f"Mean train combined:      {eval_df['train_combined'].mean():.4f}")
print(f"Mean val   SWE RMSE:      {eval_df['val_swe_rmse'].mean():.4f} mm")
print(f"Mean val   rho RMSE:      {eval_df['val_rho_rmse'].mean():.4f} kg/m³")
print(f"Mean val   combined:      {eval_df['val_combined'].mean():.4f}")

Best parameters:
  rho_max: 585.96355
  rho_null: 173.1224
  c_ov: 0.0037669391
  k_ov: 0.99310299
  k: 0.050988936
  tau: 0.05981447
  eta_null: 10664552

Train SWE  RMSE: inf mm
Train rho  RMSE: inf kg/m³
Val   SWE  RMSE: inf mm
Val   rho  RMSE: inf kg/m³
